In [1]:
# let's get some patients with graphs
from server_specific.server_utils import get_patients
import json
import numpy as np
import nibabel as nib
from scipy.ndimage import distance_transform_edt

In [2]:
patients = get_patients()
graph_patients = [patient for patient in patients if hasattr(patient, "graph_fp")]
graph_patients = graph_patients[10:20]
example_graph_patient = graph_patients[1]
example_patient_mask = nib.load(example_graph_patient.label_fp)
example_patient_mask_array = example_patient_mask.get_fdata()

In [3]:
def RAS_to_voxel(mask, point: np.array):
    # affine transformation
    point = np.linalg.inv(mask.affine) @ point
    point = point[:3]

    return point

In [4]:
def get_root_point(patient):
    graph_fp = patient.graph_fp
    
    with open(graph_fp, "r") as file:
        ground_truth_json_graph = json.load(file)
    
    gt_nodes = ground_truth_json_graph["nodes"]
    gt_roots = []

    for gt_node in gt_nodes:
        gt_root_counter = 0
        if gt_root_counter == 2:
            break

        if gt_node["is_root"]:
            gt_roots.append(gt_node)
            gt_root_counter += 1
    
    return gt_roots
        
gt_root_points = get_root_point(example_graph_patient)
print(gt_root_points)

def get_gt_voxel_coords_root_points(patient):
    gt_root_points = get_root_point(patient)
    mask = nib.load(patient.label_fp)
    
    gt_voxel_coords_root_points = []
    for gt_root_point in gt_root_points:
        pos = gt_root_point["pos"]
        pos = np.array(pos + [1])
        gt_voxel_coords_root_points.append(RAS_to_voxel(mask, point=pos))
    
    return gt_voxel_coords_root_points

gt_voxel_root_points = get_gt_voxel_coords_root_points(example_graph_patient)
print(gt_voxel_root_points)
    

[{'pos': [-23.9765625, 202.7890625, -240.54998779296875], 'is_root': True, 'id': 127}, {'pos': [-41.2265625, 162.8984375, -227.54998779296875], 'is_root': True, 'id': 201}]
[array([197., 369., 205.]), array([245., 258., 231.])]


In [5]:
def get_endpoints_in_graph(patient):
    graph_fp = patient.graph_fp
    
    with open(graph_fp, "r") as file:
        json_graph = json.load(file)
    
    endpoints = []

    nodes = json_graph["nodes"]
    # we need to sort the nodes by ids
    nodes = sorted(nodes, key=lambda x: x["id"])
    edges = json_graph["edges"]

    node_ids = [node["id"] for node in nodes]
    # we count how many times a node appears in edges.
    # if it appears only once, it's an endpoint
    bifurcation_counter = {node_id: 0 for node_id in node_ids}

    for edge in edges:
        target = edge["target"]
        source = edge["source"]

        bifurcation_counter[target] += 1
        bifurcation_counter[source] += 1
    
    # let's sort it with lowest values first
    bifurcation_counter = {k: v for k, v in sorted(bifurcation_counter.items(), key=lambda item: item[1])}
    for node_id, count in bifurcation_counter.items():
        if count > 1:
            break

        if count == 1:
            for node in nodes:
                if node.get("id") == node_id:
                    endpoints.append(node["pos"])

    return endpoints

example_patient_endpoints = get_endpoints_in_graph(example_graph_patient)

example_patient_endpoints_voxel = [RAS_to_voxel(example_patient_mask, np.array(endpoint + [1])) for endpoint in example_patient_endpoints]

In [18]:
def find_largest_points(segmentation):
    distance_transform_3d = distance_transform_edt(segmentation)
    # Find the maximum distance
    max_idx = np.unravel_index(np.argmax(distance_transform_3d), distance_transform_3d.shape)
    max_value = distance_transform_3d[max_idx]

    # print(f"Thickest point center: {max_idx}")
    
    # maybe sum up the voxels instead.
    
    # count_ones = np.count_nonzero(segmentation == 1)

    return max_idx, max_value

In [19]:
def get_patch(voxel_point, mask_array, patch_size):
    x, y, z = voxel_point
    
    shape = mask_array.shape
    x_max, y_max, z_max = shape[0], shape[1], shape[2]
    x_min = max(0, int(x-patch_size))
    x_max = min(x+patch_size, x_max)

    y_min = max(0, int(y-patch_size))
    y_max = min(y+patch_size, y_max)

    z_min = max(0, int(z-patch_size))
    z_max = min(int(z+patch_size), z_max)

    patch = mask_array[int(x_min):int(x_max), int(y_min):int(y_max), int(z_min):int(z_max)]

    return patch

def get_cutout_patches(voxel_endpoints, mask_array, patch_size):
    cutout_patches = []
    for voxel_endpoint in voxel_endpoints:
        cutout_patches.append((get_patch(voxel_endpoint, mask_array, patch_size), voxel_endpoint))
        
    return cutout_patches

In [27]:
def get_max_thickness_endpoint(endpoints, cutout_patches):
    max_value = (0, 0)
    for cutout_patch, endpoint in cutout_patches:
        # calculate the thickness score
        _, value = find_largest_points(cutout_patch)
        # print(f"value: {value}")
        if value > max_value[1]:
            max_value = [endpoint, value]
    
    return max_value

cutout_patches = get_cutout_patches(example_patient_endpoints_voxel, example_patient_mask_array, patch_size = 20)
max_thickness_endpoint, thickness = get_max_thickness_endpoint(example_patient_endpoints_voxel, cutout_patches)

print(thickness)

8.774964387392123


In [28]:
gt_voxel_root_points = get_gt_voxel_coords_root_points(example_graph_patient)
print(gt_voxel_root_points)

[array([197., 369., 205.]), array([245., 258., 231.])]


In [30]:
from skimage.measure import label, regionprops

def get_largest_connected_components(mask_array):
    # Label connected components
    labeled_array, num_features = label(mask_array, return_num=True)

    # Use a histogram to efficiently calculate the sizes of each labeled component
    component_sizes = np.bincount(labeled_array.ravel())

    # Exclude the background component (label 0)
    component_sizes[0] = 0

    # Get the two largest component labels
    largest_labels = np.argsort(component_sizes)
    largest_labels = largest_labels[-2:]

    # this should include the left and right coronary artery.
    # let's create a mask with both and compare it with the original mask

    # Create a mask for each component
    largest_mask_0 = np.isin(labeled_array, largest_labels[0])
    largest_mask_1 = np.isin(labeled_array, largest_labels[1])
    
    return largest_mask_0, largest_mask_1

largest_mask_0, largest_mask_1 = get_largest_connected_components(example_patient_mask_array)

In [31]:
def determine_root_node_for_patient(patient, patch_size):
    patient_mask = nib.load(patient.label_fp)
    patient_mask_array = patient_mask.get_fdata()
    largest_mask_0, largest_mask_1 = get_largest_connected_components(patient_mask_array)
    
    correct_endpoints_found = 0
    for largest_mask_array in [largest_mask_0, largest_mask_1]:
    # endpoints
        patient_endpoints = get_endpoints_in_graph(patient)

        # convert to voxel coordinates
        patient_endpoints_voxel = [RAS_to_voxel(patient_mask, np.array(endpoint + [1])) for endpoint in patient_endpoints]


        cutout_patches = get_cutout_patches(patient_endpoints_voxel, largest_mask_array, patch_size)
        max_thickness_endpoint, thickness = get_max_thickness_endpoint(patient_endpoints_voxel, cutout_patches)
        max_thickness_endpoint = np.array(max_thickness_endpoint)

        print(f"found rootpoint: {max_thickness_endpoint}")

        # compare with root point
        gt_voxel_root_points = get_gt_voxel_coords_root_points(patient)
        print(f"ground truth rootpoint {gt_voxel_root_points}")

        if any(np.array_equal(max_thickness_endpoint, point) for point in gt_voxel_root_points):
            correct_endpoints_found += 1
            
    return correct_endpoints_found
    
determine_root_node_for_patient(example_graph_patient, patch_size = 20)

found rootpoint: [300. 287. 205.]
ground truth rootpoint [array([197., 369., 205.]), array([245., 258., 231.])]
found rootpoint: [134. 384. 166.]
ground truth rootpoint [array([197., 369., 205.]), array([245., 258., 231.])]


0

In [34]:
patch_sizes = [2, 3, 4, 5, 7]

for patch_size in patch_sizes:
    found_nodes = 0
    print(f"patch size: {patch_size}")

    for patient in graph_patients:
        # print("I still need to do it for both")
        found_nodes += determine_root_node_for_patient(patient, patch_size)
        
    print("\n\n")
    print(f"patch_size: {patch_size}, found root nodes: {found_nodes} / {len(graph_patients) * 2}")

patch size: 2
found rootpoint: [294. 289. 204.]
ground truth rootpoint [array([196., 342., 165.]), array([233., 267., 198.])]
found rootpoint: [315. 252.  63.]
ground truth rootpoint [array([196., 342., 165.]), array([233., 267., 198.])]
found rootpoint: [336. 424.  56.]
ground truth rootpoint [array([197., 369., 205.]), array([245., 258., 231.])]
found rootpoint: [131. 387. 163.]
ground truth rootpoint [array([197., 369., 205.]), array([245., 258., 231.])]
found rootpoint: [197. 302. 171.]
ground truth rootpoint [array([197., 302., 171.]), array([240., 243., 211.])]
found rootpoint: [337. 219. 208.]
ground truth rootpoint [array([197., 302., 171.]), array([240., 243., 211.])]
found rootpoint: [282. 271.  40.]
ground truth rootpoint [array([225., 231., 190.]), array([185., 303., 162.])]
found rootpoint: [225. 231. 190.]
ground truth rootpoint [array([225., 231., 190.]), array([185., 303., 162.])]
found rootpoint: [111. 343. 138.]
ground truth rootpoint [array([193., 347., 165.]), array